In [ ]:
import pandas as pd
import wget
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
module3hw_url="https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
module3hw_data=wget.download(module3hw_url)

In [ ]:
df=pd.read_csv(module3hw_data)

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
categorical_columns = list(df.dtypes[df.dtypes == 'string'].index)

In [ ]:
categorical_columns

In [ ]:
df[categorical_columns] = df[categorical_columns].fillna('NA')

In [ ]:
df.isnull().sum()

In [ ]:
numerical_columns = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [ ]:
numerical_columns

In [ ]:
df[numerical_columns] = df[numerical_columns].fillna(0)

In [ ]:
df.isnull().sum()

In [ ]:
df.industry.mode()

In [ ]:
df[numerical_columns].corr()

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_train_wconverted = df_train.copy()
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
df_train

In [ ]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [ ]:
for feed in categorical_columns:
    res=mutual_info_score(df_train_wconverted.converted, df_train_wconverted[feed])
    print(feed, round(res, 2))

In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
model.intercept_[0]

In [ ]:
model.coef_[0].round(3)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]

In [ ]:
converted_decision = (y_pred >= 0.5)
converted_decision

In [ ]:
(y_val == converted_decision).mean()

In [ ]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision.astype(int)
df_pred['actual'] = y_val

In [ ]:
df_pred['correct'] = df_pred.prediction == df_pred.actual
df_pred.correct.mean()
origacc=df_pred.correct.mean()

In [ ]:
#exclude lead source
dv = DictVectorizer(sparse=False)
categorical_columns_wo_lead=['industry', 'employment_status', 'location']

train_dict = df_train[categorical_columns_wo_lead + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns_wo_lead + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
origacc-(y_val == converted_decision).mean()

In [ ]:
#exclude number of courses viewed
dv = DictVectorizer(sparse=False)
numerical_columns_wo_numcourse=['annual_income', 'interaction_count', 'lead_score']

train_dict = df_train[categorical_columns + numerical_columns_wo_numcourse].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns_wo_numcourse].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
origacc-(y_val == converted_decision).mean()

In [ ]:
#exclude interaction count
dv = DictVectorizer(sparse=False)
numerical_columns_wo_intcount=['annual_income', 'number_of_courses_viewed', 'lead_score']

train_dict = df_train[categorical_columns + numerical_columns_wo_intcount].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns_wo_intcount].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
origacc-(y_val == converted_decision).mean()

In [ ]:
#C=0.000001
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=0.000001, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
round((y_val == converted_decision).mean(),3)

In [ ]:
#C=0.00001
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=0.00001, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
round((y_val == converted_decision).mean(),3)

In [ ]:
#C=0.0001
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=0.0001, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
round((y_val == converted_decision).mean(),3)

In [ ]:
#C=0.001
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=0.001, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
round((y_val == converted_decision).mean(),3)